
# Mini-Project 2: Uganda Tax Engine

Course:OOP with Python
NAME: PARIYO PONYANO PRESCORT
REG No: S26M19/019 
ACCESS No: B38640

# Objective
Develop an extensible and testable Python tax engine implementing PAYE, payroll deductions, VAT, corporate income tax and a TaxPlanner extension.



In [9]:
from pathlib import Path
import sys

print("Working directory:", Path.cwd())
print("Python executable:", sys.executable)
print("src folder exists:", Path("src").is_dir())
print("tax_engine.py exists:", Path("src/tax_engine.py").is_file())

if Path("src").is_dir():
    print("Files in src:", list(Path("src").iterdir()))

Working directory: c:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\project2_uganda_tax_engine
Python executable: c:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\project2_uganda_tax_engine\.venv\Scripts\python.exe
src folder exists: True
tax_engine.py exists: True
Files in src: [WindowsPath('src/tax_engine.py'), WindowsPath('src/_init_.py'), WindowsPath('src/__pycache__')]


In [8]:
from pathlib import Path
import sys
import importlib

# Find the project folder containing the correct tax engine.
current = Path.cwd()
project_root = None

for folder in (current, *current.parents):
    tax_file = folder / "src" / "tax_engine.py"

    if tax_file.is_file():
        source = tax_file.read_text(encoding="utf-8")

        if "class Tax(" in source and "class TaxContext" in source:
            project_root = folder
            break

if project_root is None:
    raise FileNotFoundError(
        "Could not find the correct src/tax_engine.py. "
        "Check that the tax code is saved in the project root's src folder."
    )

# Put the correct project root first in Python's search path.
sys.path.insert(0, str(project_root))

# Clear previously imported modules named src or src.tax_engine.
for module_name in ("src.tax_engine", "src"):
    sys.modules.pop(module_name, None)

# Import the tax classes from the correct file.
from src.tax_engine import Money, Tax, TaxContext, TaxResult

print("Import successful!")
print("Project folder:", project_root)
print("Tax engine file:", project_root / "src" / "tax_engine.py")
print("Tax class:", Tax)

Import successful!
Project folder: c:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\project2_uganda_tax_engine
Tax engine file: c:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\project2_uganda_tax_engine\src\tax_engine.py
Tax class: <class 'src.tax_engine.Tax'>


In [7]:

from datetime import date
from decimal import Decimal

from src.tax_engine import (
    Money,
    Tax,
    TaxContext,
    PAYE,
    calculate_paye,
    calculate_payroll,
    Invoice,
    calculate_vat,
    calculate_corporate_tax,
    TaxPlanner,
    TaxScenario,
)

TAX_DATE = date(2026, 10, 1)

print("Uganda Tax Engine loaded successfully.")


Uganda Tax Engine loaded successfully.


In [6]:

from decimal import Decimal
from datetime import date
from src.tax_engine import Money, calculate_paye

# Define the tax date
TAX_DATE = date(2026, 10, 1)

# Test incomes around PAYE tax-band boundaries
boundary_incomes = [
    335000,
    335001,
    410000,
    485000,
    10_000_000,
    10_000_001,
]

for income in boundary_incomes:
    tax = calculate_paye(
        Money(Decimal(str(income))),
        residency="resident",
        tax_date=TAX_DATE,
    )
    print(f"Income: UGX {income:,.0f} | PAYE: {tax!r}")


Income: UGX 335,000 | PAYE: UGX 0
Income: UGX 335,001 | PAYE: UGX 0
Income: UGX 410,000 | PAYE: UGX 15,000
Income: UGX 485,000 | PAYE: UGX 33,750
Income: UGX 10,000,000 | PAYE: UGX 2,888,250
Income: UGX 10,000,001 | PAYE: UGX 2,888,250


In [ ]:
#COMPUTATION OF RESIDENT AND NON-RESIDENT PAYE

from decimal import Decimal
from src.tax_engine import Money, PAYE, TaxContext

# Comparing residents and non-residents
income = Money(Decimal("1000000"))

resident_context = TaxContext(
    income=income,
    residency="resident",
    tax_date="2026-10-01"
)

nonresident_context = TaxContext(
    income=income,
    residency="non-resident",
    tax_date="2026-10-01"
)

resident_tax = PAYE().calculate(resident_context)
nonresident_tax = PAYE().calculate(nonresident_context)

print("Resident PAYE:", resident_tax.amount)
print("Non-Resident PAYE:", nonresident_tax.amount)


Resident PAYE: UGX 188,250
Non-Resident PAYE: UGX 225,500


In [ ]:
#LOOKING FOR LOCATION OF payroll in tax_engine.py

import src.tax_engine as tax_engine

print("Loaded file:", tax_engine.__file__)
print("Has payroll:", hasattr(tax_engine, "payroll"))

Loaded file: c:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\project2_uganda_tax_engine\src\tax_engine.py
Has payroll: False


In [ ]:
#Ensuring payroll is defined in the tax_engine.py

import os
import sys
import src.tax_engine as tax_engine

print("Current notebook folder:")
print(os.getcwd())

print("\nPython is importing this file:")
print(tax_engine.__file__)

print("\nIs payroll defined?")
print(hasattr(tax_engine, "payroll"))

print("\nDoes the module contain Money?")
print(hasattr(tax_engine, "Money"))

Current notebook folder:
c:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\project2_uganda_tax_engine

Python is importing this file:
c:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\project2_uganda_tax_engine\src\tax_engine.py

Is payroll defined?
False

Does the module contain Money?
True


In [13]:
import src.tax_engine

print(src.tax_engine.__file__)
print(hasattr(src.tax_engine, "payroll"))

c:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\project2_uganda_tax_engine\src\tax_engine.py
False


In [ ]:
#COMPUTATION OF THE PAYROLL, PAYE, VAT, CORPORATE TAX, AND TAX SCENARIO PLANNER

from decimal import Decimal
from pathlib import Path
import sys

# Add the project root to Python's module search path
PROJECT_ROOT = Path.cwd()

# If the notebook is running from a subfolder, locate the project root
if not (PROJECT_ROOT / "src" / "tax_engine.py").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "src" / "tax_engine.py").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import from src.tax_engine
from src.tax_engine import (
    Money,
    calculate_payroll,
    calculate_paye,
    calculate_vat,
    Invoice,
    calculate_corporate_tax,
    TaxPlanner,
    TaxScenario,
)

# 1. Test payroll
result = calculate_payroll(
    gross_pay=Money(Decimal("1000000")),
    allowances=Money(Decimal("100000")),
    residency="resident",
    lst=Money(Decimal("5000")),
    tax_date="2026-10-01",
)

print("PAYROLL RESULTS")
print("Gross + allowances:", result.gross_plus_allowances)
print("Employee NSSF:", result.employee_nssf)
print("Employer NSSF:", result.employer_nssf)
print("Local Service Tax:", result.lst)
print("PAYE:", result.paye)
print("Net pay:", result.net_pay)

# 2. Test PAYE independently
paye = calculate_paye(
    Money(Decimal("1000000")),
    residency="resident",
    tax_date="2026-10-01",
)

print("\nPAYE:", paye)

# 3. Test VAT
vat = calculate_vat([
    Invoice(
        amount=Money(Decimal("100000")),
        category="standard",
        input_vat=Money(Decimal("5000")),
    )
])

print("\nVAT output:", vat.output_vat)
print("VAT input:", vat.input_vat)
print("Net VAT payable:", vat.net_payable)

# 4. Test corporate tax
chargeable, corporate_tax = calculate_corporate_tax(
    Money(Decimal("5000000")),
    Money(Decimal("2000000")),
)

print("\nChargeable income:", chargeable)
print("Corporate tax:", corporate_tax)

# 5. Test tax scenario planner
planner = TaxPlanner()

comparison = planner.compare(
    TaxScenario("Scenario A", Money(Decimal("500000"))),
    TaxScenario("Scenario B", Money(Decimal("1000000"))),
)

print("\nTAX SCENARIO COMPARISON")
print(comparison)


PAYROLL RESULTS
Gross + allowances: UGX 1,100,000
Employee NSSF: UGX 50,000
Employer NSSF: UGX 100,000
Local Service Tax: UGX 5,000
PAYE: UGX 218,250
Net pay: UGX 826,750

PAYE: UGX 188,250

VAT output: UGX 18,000
VAT input: UGX 5,000
Net VAT payable: UGX 13,000

Chargeable income: UGX 3,000,000
Corporate tax: UGX 900,000

TAX SCENARIO COMPARISON
{'first_tax': UGX 38,250, 'second_tax': UGX 188,250, 'tax_difference': UGX 150,000}


In [ ]:
#COMPUTATION OF OUTPUT VAT, INPUT VAT AND NET VAT PAYABLE
invoices = [
    Invoice(
        amount=Money(Decimal("1000000")),
        category="standard",
        input_vat=Money(Decimal("50000")),
    ),
    Invoice(
        amount=Money(Decimal("200000")),
        category="zero",
    ), 
    Invoice(
        amount=Money(Decimal("500000")),
        category="exempt", 
    ),
]

vat_result = calculate_vat(invoices)
print("Output VAT:", vat_result.output_vat.amount)
print("Input VAT:", vat_result.input_vat.amount)
print("Net VAT payable:", vat_result.net_payable.amount)


Output VAT: 180000
Input VAT: 50000
Net VAT payable: 130000


In [ ]:
#CHECKING FOR THE FILES IN THE CURRENT WORKING DIRECTORY AND PYTHON SEARCH PATHS

import os
import sys

print("Current working folder:")
print(os.getcwd())

print("\nFiles in this folder:")
print(os.listdir())

print("\nPython search paths:")
print(sys.path)


Current working folder:
c:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\project2_uganda_tax_engine

Files in this folder:
['.hypothesis', '.pytest_cache', '.venv', 'project2_uganda_tax_engine.ipynb', 'README.txt', 'requirements.txt', 'src', 'tests']

Python search paths:
['c:\\Users\\Lenovo\\Documents\\MSDS\\SEM 1\\OBJECT ORIENTED PROGRAMMING\\PYTHON PROGRAMMING\\project2_uganda_tax_engine', 'c:\\Users\\Lenovo\\Documents\\MSDS\\SEM 1\\OBJECT ORIENTED PROGRAMMING\\PYTHON PROGRAMMING\\project2_uganda_tax_engine', 'C:\\Users\\Lenovo\\AppData\\Roaming\\uv\\python\\cpython-3.14-windows-x86_64-none\\python314.zip', 'C:\\Users\\Lenovo\\AppData\\Roaming\\uv\\python\\cpython-3.14-windows-x86_64-none\\DLLs', 'C:\\Users\\Lenovo\\AppData\\Roaming\\uv\\python\\cpython-3.14-windows-x86_64-none\\Lib', 'C:\\Users\\Lenovo\\AppData\\Roaming\\uv\\python\\cpython-3.14-windows-x86_64-none', 'c:\\Users\\Lenovo\\Documents\\MSDS\\SEM 1\\OBJECT ORIENTED PROGRAMMING\\PYTHON P

In [ ]:

from pathlib import Path

root = Path.home() / "Documents" / "MSDS"

matches = list(root.rglob("tax_engine.py"))

if matches:
    for file in matches:
        print("Found:", file)
else:
    print("tax_engine.py was not found in your MSDS folder.")


Found: C:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\Weather forecast\src\tax_engine.py
Found: C:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\project2_uganda_tax_engine\src\tax_engine.py


In [ ]:
#COMPUTATION OF THE PAYROLL, PAYE, VAT, CORPORATE TAX, AND TAX SCENARIO PLANNER


from decimal import Decimal
from src.tax_engine import Money, calculate_payroll

result = calculate_payroll(
    gross_pay=Money(Decimal("1000000")),
    allowances=Money(Decimal("100000")),
    residency="resident",
    lst=Money(Decimal("5000")),
    tax_date="2026-10-01",
)

print("Result type:", type(result))
print("Payroll result:", result)


Result type: <class 'src.tax_engine.PayrollResult'>
Payroll result: PayrollResult(gross_plus_allowances=UGX 1,100,000, employee_nssf=UGX 50,000, employer_nssf=UGX 100,000, lst=UGX 5,000, paye=UGX 218,250, net_pay=UGX 826,750)


In [19]:

from pathlib import Path
import os
import sys

print("Current notebook folder:")
print(Path.cwd())

print("\nPROJECT_FOLDER value:")
print(globals().get("PROJECT_FOLDER", "Not defined"))

print("\nDoes tax_engine.py exist in the current folder?")
print((Path.cwd() / "tax_engine.py").exists())

print("\nPython search paths:")
for path in sys.path[:10]:
    print(path)

print("\nSearching Documents/MSDS for tax_engine.py:")

search_folder = Path.home() / "Documents" / "MSDS"

if search_folder.exists():
    matches = list(search_folder.rglob("tax_engine.py"))

    if matches:
        for file in matches:
            print("FOUND:", file)
    else:
        print("No tax_engine.py file found in Documents/MSDS.")
else:
    print("Documents/MSDS folder does not exist.")


Current notebook folder:
c:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\project2_uganda_tax_engine

PROJECT_FOLDER value:
Not defined

Does tax_engine.py exist in the current folder?
False

Python search paths:
c:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\project2_uganda_tax_engine
c:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\project2_uganda_tax_engine
C:\Users\Lenovo\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\python314.zip
C:\Users\Lenovo\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\DLLs
C:\Users\Lenovo\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib
C:\Users\Lenovo\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none
c:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\project2_uganda_tax_engine\.venv

c:\Users\Lenovo\Documents\MSDS\SEM 1\OBJECT ORIENTED PROGRAMMING\PYTHON PROGRAMMING\proje

In [ ]:
#IMPORTING THE TAX ENGINE FROM THE SRC FOLDER
import sys
from pathlib import Path
from decimal import Decimal

# Your project folder
PROJECT_FOLDER = Path.cwd()

# The folder containing tax_engine.py
SRC_FOLDER = PROJECT_FOLDER / "src"

# Confirm the file exists
assert (SRC_FOLDER / "tax_engine.py").is_file(), (
    f"tax_engine.py was not found in {SRC_FOLDER}"
)

# Add src to Python's search path
if str(SRC_FOLDER) not in sys.path:
    sys.path.insert(0, str(SRC_FOLDER))

# Import your tax engine
from tax_engine import (
    Money,
    calculate_payroll,
    calculate_paye,
    calculate_vat,
    Invoice,
    calculate_corporate_tax,
    TaxPlanner,
    TaxScenario,
)

print("Tax engine imported successfully!")


Tax engine imported successfully!
